In [ ]:
import os
import json
import random
import time
import pandas as pd
from pydantic import BaseModel, model_validator, ValidationError


In [ ]:
# ==========================================
# 1. SYMBOLIC LAYER (PYDANTIC DEFINITION)
# ==========================================
class InvoiceSummary(BaseModel):
    total_net_worth: float
    total_vat: float
    total_gross_worth: float

class Invoice(BaseModel):
    invoice_no: str
    client_name: str
    summary: InvoiceSummary

    @model_validator(mode='after')
    def check_mathematical_integrity(self):
        """Symbolic rule: Net Worth + VAT must exactly equal Gross Worth."""
        expected_gross = round(self.summary.total_net_worth + self.summary.total_vat, 2)
        if round(self.summary.total_gross_worth, 2) != expected_gross:
            raise ValueError(
                f"Math Failure: Net ({self.summary.total_net_worth}) + "
                f"VAT ({self.summary.total_vat}) != Gross ({self.summary.total_gross_worth})"
            )
        return self



In [ ]:
# ==========================================
# 2. DATA INGESTION (KAGGLE DATASET)
# ==========================================
def load_kaggle_data():
    """Loads batch_1.csv if available; otherwise mocks the schema for execution."""
    if os.path.exists('batch_1.csv'):
        return pd.read_csv('batch_1.csv')
    
    print("Notice: batch_1.csv not found locally. Generating synthetic mock matching the Kaggle schema for the test run...\n")
    mock_data = []
    for i in range(100):
        net = round(random.uniform(10000, 150000), 2)
        vat = round(net * 0.10, 2) # 10% uniform VAT per dataset specs
        gross = round(net + vat, 2)
        
        ground_truth_json = {
            "invoice_no": str(51109301 + i),
            "client": {"name": f"Tech Client {i}"},
            "summary": {
                "total_net_worth": net,
                "total_vat": vat,
                "total_gross_worth": gross
            }
        }
        
        mock_data.append({
            "filename": f"invoice_{51109301 + i}.pdf",
            "ocred_text": f"Invoice No: {51109301 + i}\nClient: Tech Client {i}\n...[unstructured items]...\nNet Worth: {net} INR\nVAT (10%): {vat} INR\nTotal Gross: {gross} INR",
            "json_data": json.dumps(ground_truth_json)
        })
    return pd.DataFrame(mock_data)



In [ ]:
# ==========================================
# 3. NEURAL LAYER (SIMULATED LLM)
# ==========================================
def simulate_llm_extraction(row):
    """
    Simulates extracting JSON from the Kaggle ocred_text.
    Injects 5% syntax errors and 10% hallucinated VAT/Math drift.
    """
    time.sleep(0.01) # Simulating API latency
    truth = json.loads(row['json_data'])
    rand_val = random.random()
    
    if rand_val < 0.05:
        # Missing closing brace to simulate syntax crash
        return f'{{"invoice_no": "{truth["invoice_no"]}", "client_name": "{truth["client"]["name"]}", "summary": {{"total_net_worth": {truth["summary"]["total_net_worth"]}, "total_vat": {truth["summary"]["total_vat"]}, "total_gross_worth": {truth["summary"]["total_gross_worth"]}}}'
    
    elif rand_val < 0.15:
        # LLM hallucination: slightly hallucinates the VAT or Gross total
        bad_gross = round(truth["summary"]["total_gross_worth"] + random.uniform(500, 2000), 2)
        return json.dumps({
            "invoice_no": truth["invoice_no"],
            "client_name": truth["client"]["name"],
            "summary": {
                "total_net_worth": truth["summary"]["total_net_worth"],
                "total_vat": truth["summary"]["total_vat"],
                "total_gross_worth": bad_gross
            }
        })
    
    else:
        # Perfect extraction
        return json.dumps({
            "invoice_no": truth["invoice_no"],
            "client_name": truth["client"]["name"],
            "summary": {
                "total_net_worth": truth["summary"]["total_net_worth"],
                "total_vat": truth["summary"]["total_vat"],
                "total_gross_worth": truth["summary"]["total_gross_worth"]
            }
        })



In [ ]:
# ==========================================
# 4. EXECUTION AND EVALUATION
# ==========================================
def run_evaluation():
    df = load_kaggle_data()
    
    metrics = {
        "pure_llm": {"success": 0, "syntax_errors": 0, "silent_math_errors": 0},
        "neuro_symbolic": {"success": 0, "syntax_errors_caught": 0, "logic_errors_caught": 0}
    }

    print("Evaluating Pure LLM Architecture...")
    for _, row in df.iterrows():
        raw_output = simulate_llm_extraction(row)
        try:
            parsed = json.loads(raw_output)
            # Pure LLM takes the data blindly. Check if it was secretly mathematically poisoned.
            expected_gross = round(parsed['summary']['total_net_worth'] + parsed['summary']['total_vat'], 2)
            if round(parsed['summary']['total_gross_worth'], 2) != expected_gross:
                metrics["pure_llm"]["silent_math_errors"] += 1
            else:
                metrics["pure_llm"]["success"] += 1
        except json.JSONDecodeError:
            metrics["pure_llm"]["syntax_errors"] += 1

    print("Evaluating Hybrid Neuro-Symbolic Architecture...")
    for _, row in df.iterrows():
        raw_output = simulate_llm_extraction(row)
        try:
            parsed = json.loads(raw_output)
            # Symbolic validation step
            validated_invoice = Invoice(**parsed) 
            metrics["neuro_symbolic"]["success"] += 1
        except json.JSONDecodeError:
            metrics["neuro_symbolic"]["syntax_errors_caught"] += 1
        except ValidationError as e:
            metrics["neuro_symbolic"]["logic_errors_caught"] += 1

    return metrics

results = run_evaluation()

print("\n==================================================")
print("     DSR EVALUATION RESULTS (KAGGLE DATASET)      ")
print("==================================================")
print(f"Dataset Size: 100 OCR-Ready Invoices (Consumer Electronics)\n")
print("[BASELINE] PURE NEURAL ARCHITECTURE:")
print(f"   - Clean Extractions:       {results['pure_llm']['success']}")
print(f"   - Parsing Crashes:         {results['pure_llm']['syntax_errors']}")
print(f"   - Poisoned Data Ingested:  {results['pure_llm']['silent_math_errors']} (Silent Math Failures)\n")

print("[PROPOSED] HYBRID NEURO-SYMBOLIC ARCHITECTURE:")
print(f"   - Validated Extractions:   {results['neuro_symbolic']['success']}")
print(f"   - Parsing Crashes Handled: {results['neuro_symbolic']['syntax_errors_caught']}")
print(f"   - Hallucinations Blocked:  {results['neuro_symbolic']['logic_errors_caught']} (Prevented Data Poisoning)")
print("==================================================")